# Pipeline & Practice

Learn how to integrate preprocessing and modeling into a single workflow using sklearn's Pipeline and ColumnTransformer.

**Learning Objectives:**
- Understand the necessity and benefits of Pipelines
- Process different feature types with ColumnTransformer
- Write custom Transformers
- Combine Pipeline with GridSearchCV
- Model saving and deployment

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.datasets import load_iris, load_breast_cancer

import warnings
warnings.filterwarnings('ignore')

## 1. Pipeline Basics

### Problems Without Using Pipeline

1. **Data Leakage**: Risk of test data information leaking into training
2. **Code Complexity**: Multiple steps must be managed manually
3. **Reproducibility Issues**: Possible order mistakes and parameter mismatches

### Benefits of Pipeline

1. Code simplification
2. Data leakage prevention
3. Perfect integration with cross-validation
4. Easy hyperparameter tuning
5. Convenient model saving/deployment

In [ ]:
# Load data
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

# Create Pipeline (explicit names)
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=2)),
    ('classifier', LogisticRegression())
])

# Train and predict
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
score = pipeline.score(X_test, y_test)

print(f"Pipeline accuracy: {score:.4f}")

# make_pipeline (automatic name generation)
pipeline_auto = make_pipeline(
    StandardScaler(),
    PCA(n_components=2),
    LogisticRegression()
)

pipeline_auto.fit(X_train, y_train)
print(f"make_pipeline accuracy: {pipeline_auto.score(X_test, y_test):.4f}")

### Accessing Pipeline Steps

In [ ]:
# Check step names
print("Pipeline steps:")
for name, step in pipeline.named_steps.items():
    print(f"  {name}: {type(step).__name__}")

# Access specific steps
print(f"\nPCA explained variance: {pipeline.named_steps['pca'].explained_variance_ratio_}")
print(f"Logistic regression coefficient shape: {pipeline.named_steps['classifier'].coef_.shape}")

# Get intermediate step results
X_scaled = pipeline.named_steps['scaler'].transform(X_test)
X_pca = pipeline.named_steps['pca'].transform(X_scaled)
print(f"\nShape after scaling: {X_scaled.shape}")
print(f"Shape after PCA: {X_pca.shape}")

## 2. ColumnTransformer - Processing Different Feature Types

Real-world data contains a mix of numeric and categorical features. ColumnTransformer allows you to apply appropriate preprocessing to each type.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier

# Generate sample data
data = {
    'age': [25, 32, 47, 51, 62, 28, 35, 42, 55, 60],
    'income': [50000, 60000, 80000, 120000, 95000, 55000, 70000, 85000, 110000, 100000],
    'gender': ['M', 'F', 'M', 'F', 'M', 'F', 'M', 'F', 'M', 'F'],
    'education': ['Bachelor', 'Master', 'PhD', 'Bachelor', 'Master', 
                  'Bachelor', 'PhD', 'Master', 'PhD', 'Bachelor'],
    'purchased': [0, 1, 1, 1, 0, 0, 1, 1, 1, 0]
}
df = pd.DataFrame(data)

X = df.drop('purchased', axis=1)
y = df['purchased']

print("Data types:")
print(X.dtypes)
print("\nData sample:")
print(X.head())

In [ ]:
# Feature classification
numeric_features = ['age', 'income']
categorical_features = ['gender', 'education']

# Why: ColumnTransformer applies different preprocessing to different column types
# in a single step — numerical features need scaling while categorical features
# need encoding, and mixing these transformations would be incorrect.
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        # Why: drop='first' avoids the dummy variable trap (perfect multicollinearity)
        # which can destabilize linear models.
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
    ],
    remainder='passthrough'  # Handle remaining features: 'drop' or 'passthrough'
)

# Transform
X_transformed = preprocessor.fit_transform(X)

print(f"Original shape: {X.shape}")
print(f"Transformed shape: {X_transformed.shape}")

# Transformed feature names
feature_names = (
    numeric_features +
    list(preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features))
)
print(f"\nFeature names: {feature_names}")

### Combining Pipeline + ColumnTransformer

In [ ]:
# Full pipeline
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Train
full_pipeline.fit(X, y)

# Predict with new data
new_data = pd.DataFrame({
    'age': [30, 45],
    'income': [70000, 90000],
    'gender': ['F', 'M'],
    'education': ['Master', 'PhD']
})

predictions = full_pipeline.predict(new_data)
print(f"Predictions: {predictions}")

## 3. Complex Pipeline with Missing Value Handling

In [ ]:
from sklearn.impute import SimpleImputer

# Generate data with missing values
data_missing = {
    'age': [25, np.nan, 47, 51, 62, 28, np.nan, 42, 55, 60],
    'income': [50000, 60000, np.nan, 120000, 95000, np.nan, 70000, 85000, 110000, 100000],
    'gender': ['M', 'F', 'M', None, 'M', 'F', 'M', None, 'M', 'F'],
    'education': ['Bachelor', 'Master', 'PhD', 'Bachelor', None, 
                  'Bachelor', 'PhD', 'Master', None, 'Bachelor'],
    'purchased': [0, 1, 1, 1, 0, 0, 1, 1, 1, 0]
}
df_missing = pd.DataFrame(data_missing)
X_missing = df_missing.drop('purchased', axis=1)
y_missing = df_missing['purchased']

print("Missing value counts:")
print(X_missing.isnull().sum())

In [ ]:
# Why: Imputing numerics with median is robust to outliers (unlike mean), while
# categorical features use most_frequent since there is no meaningful "average" for categories.
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline (with missing value handling)
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    # Why: handle_unknown='ignore' prevents errors at prediction time when the model
    # encounters category values not seen during training.
    ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

# ColumnTransformer
preprocessor_full = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Full pipeline
complete_pipeline = Pipeline([
    ('preprocessor', preprocessor_full),
    ('classifier', RandomForestClassifier(random_state=42))
])

complete_pipeline.fit(X_missing, y_missing)
print("Pipeline with missing values trained successfully")
print(f"Training accuracy: {complete_pipeline.score(X_missing, y_missing):.4f}")

## 4. Pipeline with Cross-Validation and Hyperparameter Tuning

In [ ]:
# Real dataset for practice
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

# Define pipeline
pipeline_cv = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Why: When cross_val_score wraps a Pipeline, each fold independently fits the scaler
# on training data only, preventing data leakage that would occur if you scaled the
# entire dataset before splitting.
scores = cross_val_score(pipeline_cv, X, y, cv=5, scoring='accuracy')

print("Cross-validation results:")
print(f"  Each fold: {scores}")
print(f"  Mean: {scores.mean():.4f} (+/- {scores.std():.4f})")

### Hyperparameter Tuning with GridSearchCV

In Pipelines, hyperparameter names use the `step__parameter` format.

In [ ]:
# Parameter grid (step__parameter format)
# Why: Including the scaler itself as a hyperparameter lets GridSearch determine
# which scaling method works best for this model and data combination.
param_grid = {
    'scaler': [StandardScaler(), MinMaxScaler()],
    'classifier__C': [0.1, 1, 10],
    'classifier__penalty': ['l1', 'l2'],
    'classifier__solver': ['liblinear']
}

# Grid Search
grid_search = GridSearchCV(
    pipeline_cv,
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X, y)

print("\nGrid Search Results:")
print(f"  Best parameters: {grid_search.best_params_}")
print(f"  Best score: {grid_search.best_score_:.4f}")

### Comparing Multiple Models

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Pipeline for multiple models
pipeline_multi = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression())  # placeholder
])

# Different parameters for each model
param_grid_multi = [
    {
        'classifier': [LogisticRegression(max_iter=1000)],
        'classifier__C': [0.1, 1, 10]
    },
    {
        'classifier': [RandomForestClassifier(random_state=42)],
        'classifier__n_estimators': [50, 100],
        'classifier__max_depth': [None, 5, 10]
    },
    {
        'classifier': [SVC()],
        'classifier__C': [0.1, 1],
        'classifier__kernel': ['rbf', 'linear']
    }
]

grid_search_multi = GridSearchCV(
    pipeline_multi,
    param_grid_multi,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search_multi.fit(X, y)

print("\nMultiple Model Comparison Results:")
print(f"  Best model: {type(grid_search_multi.best_params_['classifier']).__name__}")
print(f"  Best parameters: {grid_search_multi.best_params_}")
print(f"  Best score: {grid_search_multi.best_score_:.4f}")

## 5. Model Saving and Loading

Trained pipelines can be saved and reused later.

In [ ]:
import joblib
import pickle
import sklearn
from datetime import datetime

# Best model
best_pipeline = grid_search.best_estimator_

# 1. Save with joblib (recommended)
joblib.dump(best_pipeline, 'best_model.joblib')
print("Model saved: best_model.joblib")

# Load model
loaded_model = joblib.load('best_model.joblib')

# Test
X_test_sample = X[:5]
predictions = loaded_model.predict(X_test_sample)
print(f"Loaded model predictions: {predictions}")

In [ ]:
# 2. Save with pickle
with open('model.pkl', 'wb') as f:
    pickle.dump(best_pipeline, f)

# Load pickle
with open('model.pkl', 'rb') as f:
    loaded_model_pkl = pickle.load(f)

print("Pickle model prediction:", loaded_model_pkl.predict(X[:3]))

### Saving with Metadata (Recommended)

In [ ]:
# Why: Saving metadata alongside the model ensures reproducibility — you can verify
# the sklearn version matches at load time and trace back which parameters produced
# which performance, critical for production model management.
model_metadata = {
    'model': best_pipeline,
    'sklearn_version': sklearn.__version__,
    'training_date': datetime.now().isoformat(),
    'feature_names': list(cancer.feature_names),
    'target_names': list(cancer.target_names),
    'cv_score': grid_search.best_score_,
    'best_params': grid_search.best_params_
}

joblib.dump(model_metadata, 'model_with_metadata.joblib')

# Load and verify
loaded_metadata = joblib.load('model_with_metadata.joblib')
print("Model Metadata:")
print(f"  Training date: {loaded_metadata['training_date']}")
print(f"  sklearn version: {loaded_metadata['sklearn_version']}")
print(f"  CV score: {loaded_metadata['cv_score']:.4f}")
print(f"  Best parameters: {loaded_metadata['best_params']}")

## 6. Writing Custom Transformers

You can create your own Transformers by inheriting from sklearn's BaseEstimator and TransformerMixin.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class OutlierRemover(BaseEstimator, TransformerMixin):
    """Transformer that replaces outliers with boundary values"""
    
    def __init__(self, threshold=3):
        self.threshold = threshold
        self.mean_ = None
        self.std_ = None
    
    def fit(self, X, y=None):
        self.mean_ = np.mean(X, axis=0)
        self.std_ = np.std(X, axis=0)
        return self
    
    def transform(self, X):
        X = np.array(X)
        z_scores = np.abs((X - self.mean_) / (self.std_ + 1e-10))
        # Replace outliers with boundary values
        X_clipped = np.where(z_scores > self.threshold,
                             self.mean_ + self.threshold * self.std_ * np.sign(X - self.mean_),
                             X)
        return X_clipped


class FeatureSelector(BaseEstimator, TransformerMixin):
    """Feature selection transformer"""
    
    def __init__(self, feature_indices=None):
        self.feature_indices = feature_indices
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = np.array(X)
        if self.feature_indices is not None:
            return X[:, self.feature_indices]
        return X


# Using custom transformers
custom_pipeline = Pipeline([
    ('outlier', OutlierRemover(threshold=3)),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000))
])

scores = cross_val_score(custom_pipeline, X, y, cv=5)
print(f"Custom transformer CV score: {scores.mean():.4f} (+/- {scores.std():.4f})")

## 7. Practical Template - Classification Pipeline Builder Function

In [ ]:
from sklearn.compose import make_column_selector

def create_classification_pipeline(model, numeric_features=None, categorical_features=None):
    """
    Create a classification pipeline.
    
    Parameters:
    -----------
    model : sklearn estimator
        Classification model
    numeric_features : list, optional
        List of numeric feature names
    categorical_features : list, optional
        List of categorical feature names
    
    Returns:
    --------
    pipeline : Pipeline
        Preprocessing + model pipeline
    """
    
    # Numeric feature pipeline
    numeric_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    
    # Categorical feature pipeline
    categorical_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
    ])
    
    # ColumnTransformer
    if numeric_features is None and categorical_features is None:
        # Auto-detect
        preprocessor = ColumnTransformer(
            transformers=[
                ('num', numeric_transformer, make_column_selector(dtype_include=np.number)),
                ('cat', categorical_transformer, make_column_selector(dtype_include=object))
            ]
        )
    else:
        preprocessor = ColumnTransformer(
            transformers=[
                ('num', numeric_transformer, numeric_features or []),
                ('cat', categorical_transformer, categorical_features or [])
            ]
        )
    
    # Full pipeline
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    return pipeline


# Usage example
from sklearn.ensemble import GradientBoostingClassifier

pipeline_template = create_classification_pipeline(
    GradientBoostingClassifier(random_state=42),
    numeric_features=['age', 'income'],
    categorical_features=['gender', 'education']
)

print("Classification pipeline template created")
print(pipeline_template)

## 8. Model Wrapper Class for Deployment

In [ ]:
class ModelWrapper:
    """Model wrapper for deployment"""
    
    def __init__(self, model_path):
        self.model = joblib.load(model_path)
        self.feature_names = None
    
    def set_feature_names(self, names):
        """Set feature names"""
        self.feature_names = names
    
    def predict(self, input_data):
        """Handle dictionary or DataFrame input"""
        if isinstance(input_data, dict):
            input_data = pd.DataFrame([input_data])
        
        if self.feature_names:
            input_data = input_data[self.feature_names]
        
        return self.model.predict(input_data)
    
    def predict_proba(self, input_data):
        """Probability prediction"""
        if isinstance(input_data, dict):
            input_data = pd.DataFrame([input_data])
        
        if self.feature_names:
            input_data = input_data[self.feature_names]
        
        return self.model.predict_proba(input_data)


# Usage example
# wrapper = ModelWrapper('best_model.joblib')
# wrapper.set_feature_names(cancer.feature_names)
# prediction = wrapper.predict(X[0:1])
# print(f"Prediction: {prediction}")

## Summary and Best Practices

### Benefits of Using Pipeline

1. **Data leakage prevention**: During cross-validation, preprocessing is performed using only training data in each fold
2. **Code simplification**: Manage multiple steps as a single object
3. **Reproducibility**: All preprocessing steps are saved, ensuring identical processing
4. **Easy deployment**: Save the entire workflow as a single file

### Hyperparameter Naming Convention

```python
# Format: step_name__parameter_name
'classifier__C'  # C parameter of the classifier step
'preprocessor__num__scaler__with_mean'  # Nested parameters
```

### Model Saving Method Comparison

| Method | Advantages | Disadvantages |
|--------|-----------|---------------|
| joblib | Efficient for large NumPy arrays | sklearn-specific |
| pickle | Python standard library | Slow for large data |
| ONNX | Framework-independent, multi-language support | Conversion work required |

### Practical Checklist

- [ ] Always use Pipeline to prevent data leakage
- [ ] Use ColumnTransformer to separate numeric/categorical preprocessing
- [ ] Include metadata when saving models (version, date, performance, etc.)
- [ ] Write input validation functions
- [ ] Custom Transformers should inherit from BaseEstimator and TransformerMixin
- [ ] Tune the entire pipeline with GridSearchCV